# Pacotes

In [1]:
import pandas as pd

from sklearn import  model_selection
from sklearn.tree import DecisionTreeClassifier
from scipy import stats
from sklearn.model_selection import RandomizedSearchCV 
from sklearn.metrics import confusion_matrix

from sklearn import tree
from matplotlib import pyplot as plt

from sklearn.model_selection import KFold 
from sklearn.model_selection import cross_val_score 

import numpy as np
import seaborn as sns
from prettytable import PrettyTable
import sklearn

# Modelo

In [5]:
for i in range(18):
    
    print('')
    print('#_______________ Arquivo ', i, '________________#')
    print('')
    
    i=str(i)

    
    # Extração dos dados
    
    X_treino = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_ins_und.csv'
                     ,sep = ','
                     ,decimal='.'
                      )
    
    print('')
    print('Tamanho do treino e quantidade de ocorrências')
    print(len(X_treino.index))
    print(round(sum(X_treino['flg_ocorrencia'])))
    print('')
    
    X_teste = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_oos.csv'
                     ,sep = ','
                     ,decimal='.'
                      )
    
    print('')
    print('Tamanho do teste e quantidade de ocorrências')
    print(len(X_teste.index))
    print(round(sum(X_teste['flg_ocorrencia'])))
    print('')
    
    oot = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_final_oot.csv'
                     ,sep = ','
                     ,decimal='.'
                      )
    
    print('')
    print('Tamanho do teste 2 meses a frente e quantidade de ocorrências')
    print(len(oot.index))
    print(round(sum(oot['flg_ocorrencia'])))
    print('')
    
    if round(sum(oot['flg_ocorrencia'])) == 0 :
        continue
    
    #Ajuste na flg_ocorrencia
    
    y_treino = X_treino['flg_ocorrencia']
    X_treino = X_treino.drop('flg_ocorrencia', axis=1)
    n = len(X_treino.columns)
    
    y_teste = X_teste['flg_ocorrencia']
    X_teste = X_teste.drop('flg_ocorrencia', axis=1)
    
    ###########################################
    ############ Árvore de Decisão ############
    ###########################################
    
    maxi = int(sum(y_treino)-1) if sum(y_treino) < 10 else 10
    min_s = int((sum(y_treino)/10)+1) if sum(y_treino) < 300 else 30
    
    classificador_arvore = DecisionTreeClassifier()

    lista_parametros = {'criterion': ['gini', 'entropy', 'log_loss'],
        'splitter': ['best', 'random'],
        'max_depth': [i + 1 for i in range(2,30,1)],
        'min_samples_split': [i + 1 for i in range(1,int(maxi-1),1)],
        'min_impurity_decrease': stats.uniform(0, 1)
        }

    rand_search = RandomizedSearchCV(classificador_arvore, 
                                     param_distributions = lista_parametros,
                                     cv = int(maxi-1),
                                     random_state = 42,
                                     scoring = 'f1') 
    rand_search.fit(X_treino.iloc[:,3:n], y_treino) 
    
    print('')
    print(rand_search.best_estimator_)
    print('')
    
    classificador_arvore = rand_search.best_estimator_
    classificador_arvore.fit(X_treino.iloc[:,3:n], y_treino) 
    
    print('')
    text_representation = tree.export_text(classificador_arvore, feature_names = list(X_treino.iloc[:,3:n].columns) )
    print(text_representation)
    print('')
    
    #############################################
    ############ Qualidade do modelo ############
    #############################################
    
    kfold  = KFold(n_splits=10, shuffle=False)
    result_arvore = cross_val_score(classificador_arvore, X_treino.iloc[:,3:n], y_treino, cv = kfold)
    #K-fold
    #'Média da acurácia:'
    print(np.mean(result_arvore))
    #'Variância da acurácia'
    print(np.std(result_arvore))

    #Teste - OOS
    y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])
    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])),2))

    #Teste - OOT
    y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])
    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(oot.iloc[:,n], classificador_arvore.predict(oot.iloc[:,4:n+1])),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(oot.iloc[:,n], classificador_arvore.predict(oot.iloc[:,4:n+1])),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(oot.iloc[:,n], classificador_arvore.predict(oot.iloc[:,4:n+1])),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(oot.iloc[:,n], classificador_arvore.predict(oot.iloc[:,4:n+1])),2))


#_______________ Arquivo  0 ________________#


Tamanho do treino e quantidade de ocorrências
18
9


Tamanho do teste e quantidade de ocorrências
1824
3


Tamanho do teste 2 meses a frente e quantidade de ocorrências
24890
35


DecisionTreeClassifier(criterion='entropy', max_depth=24,
                       min_impurity_decrease=0.007066305219717406)


|--- status_noventaseishoras <= 4.00
|   |--- ADD_divisores_talvegues <= 7.50
|   |   |--- noventaseishoras <= 50.40
|   |   |   |--- mes <= 19.60
|   |   |   |   |--- class: 1.0
|   |   |   |--- mes >  19.60
|   |   |   |   |--- class: 0.0
|   |   |--- noventaseishoras >  50.40
|   |   |   |--- class: 1.0
|   |--- ADD_divisores_talvegues >  7.50
|   |   |--- class: 1.0
|--- status_noventaseishoras >  4.00
|   |--- class: 0.0


0.7
0.4
0.55
0.0
1.0
0.01
0.68
1.0
0.68
0.81

#_______________ Arquivo  1 ________________#


Tamanho do treino e quantidade de ocorrências
62
31


Tamanho do teste e quantidade de ocorrências
8301
16


Tamanho d

0.47
0.0
0.94
0.0
0.95
0.97
0.97
0.97

#_______________ Arquivo  13 ________________#


Tamanho do treino e quantidade de ocorrências
850
425


Tamanho do teste e quantidade de ocorrências
212133
239


Tamanho do teste 2 meses a frente e quantidade de ocorrências
80
1


DecisionTreeClassifier(max_depth=9, min_impurity_decrease=0.013264961159866528,
                       splitter='random')


|--- flg_comunidades <= 0.75
|   |--- flg_cobertura_vegetal <= 0.99
|   |   |--- flg_Declividade_numerica_maior_10 <= 0.15
|   |   |   |--- class: 0.0
|   |   |--- flg_Declividade_numerica_maior_10 >  0.15
|   |   |   |--- class: 1.0
|   |--- flg_cobertura_vegetal >  0.99
|   |   |--- class: 0.0
|--- flg_comunidades >  0.75
|   |--- class: 1.0


0.5929411764705883
0.1669759529337428
0.46
0.0
0.94
0.0
1.0
1.0
1.0
1.0

#_______________ Arquivo  14 ________________#


Tamanho do treino e quantidade de ocorrências
852
426


Tamanho do teste e quantidade de ocorrências
212537
239


Tamanho do teste 2 me